# Вступ до роботи з локальними LLM

Цей ноутбук — практичний воркшоп про роботу з локальними LLM та їх порівняння з хмарними моделями (на прикладі OpenAI).

## Що розглянемо

1. **Локальні LLM: що це і коли варто.** Локальна LLM працює на вашому залізі, без зовнішнього API. Це має сенс, коли важлива приватність даних, немає стабільного доступу до інтернету, або вартість хмарного API стає критичною на масштабі.
2. **Ollama та альтернативи.** Основний інструмент воркшопу — Ollama. Коротко розглянемо й альтернативи: Hugging Face, Docker Model Runner та vLLM.
3. **Як вибрати модель: бенчмарк exact-match.** Порівняємо Ollama та OpenAI на задачах з однозначною правильною відповіддю (MMLU), де точність рахується автоматично.
4. **LLM-as-a-Judge.** Порівняємо відповіді моделей на відкритих питаннях, де немає єдиної "правильної" відповіді, за допомогою окремої LLM-судді.
5. **Готові фреймворки для оцінювання.** Подивимось на існуючі інструменти (RAGAS, DeepEval, TruLens, OpenAI Evals), які позбавляють потреби писати кожну метрику з нуля.
6. **Готові дашборди і leaderboard-и.** Розглянемо, де дивитися актуальні публічні порівняння моделей (LLM Stats, Vellum).

Це невеликий технічний воркшоп, а не повноцінна оцінка LLM. Для продакшн-рішень варто орієнтуватися на задачі саме вашого реального use case.


------
## 000 - Підготовка: робота з віртуальним середовищем (venv)

### Для чого потрібне віртуальне середовище
- Ізолює залежності проєкту від глобального Python у системі.
- Дозволяє різним проєктам мати різні версії бібліотек без конфліктів.
- Полегшує відтворюваність: у команді всі ставлять пакети в однакове середовище.

### Створення віртуального середовища
```bash
python3 -m venv .venv
```

### Активація віртуального середовища
macOS / Linux:
```bash
source .venv/bin/activate
```

Windows (PowerShell):
```powershell
.venv\Scripts\Activate.ps1
```

Windows (cmd):
```cmd
.venv\Scripts\activate.bat
```

### Перевірка та вихід
Перевірити, що активовано:
```bash
which python
```

Вийти з середовища:
```bash
deactivate
```
------

## Intoduction
---------

## 1. Локальні LLM: що це і коли варто.
Локальна LLM працює на вашому залізі, без зовнішнього API. Це має сенс, коли важлива приватність даних, немає стабільного доступу до інтернету, або вартість хмарного API стає критичною на масштабі.

## 1.2. Ollama

**Що таке Ollama?**
- Ollama — це локальний runtime для LLM-моделей.
- Модель запускається на вашому комп'ютері, без зовнішнього LLM API.
- Для цього ноутбука **акаунт Ollama не потрібен**.

**Як встановити Ollama**
- Відкрийте офіційний сайт: https://ollama.com/download
- Завантажте версію під вашу ОС (macOS / Windows / Linux).
- Встановіть застосунок і запустіть Ollama.

**Базові операції з Ollama**

Після запуску Ollama працюємо через Terminal:
- Перевірка вже завантажених моделей: `ollama list`
- Завантажити нову модель LLM: `ollama pull qwen2.5:3b-instruct` (~1.9GB, ~4 хв).
- Завантажити нову embedding-модель: `ollama pull nomic-embed-text` (~274MB, ~30 сек).
- Швидкий тест моделі: `ollama run qwen2.5:3b-instruct`
  і задайте просте питання, наприклад: `Hi! How many countries exist?`

**Де подивитися інші моделі**
- Відкрийте бібліотеку моделей Ollama: https://ollama.com/library
- Там є назви моделей і теги, які можна завантажувати через `ollama pull <model:tag>`.

### Перед стартом
- Переконайтесь, що Ollama запущений локально.
- Переконайтесь, що модель уже завантажена (приклад: `qwen2.5:3b-instruct`).

## 1.3. Aльтернативні способи запуску LLM локально

**Hugging Face (коротко, як альтернатива)**
- Hugging Face — це екосистема моделей, датасетів і бібліотек: https://huggingface.co/
- Через `transformers`/`huggingface_hub` можна запускати моделі локально або через Inference API, але це вимагає більше налаштувань (токени, вибір runtime, підбір заліза під модель).
- У цій сесії ми будуємо все на Ollama, а HF тримаємо як потенційну альтернативу для самостійних експериментів.

**Docker Model Runner та vLLM (дуже коротко)**
- **Docker Model Runner** — запуск моделей у контейнерах Docker. 
- **vLLM** — швидкий inference-сервер для LLM, орієнтований на продакшн-навантаження.
- Обидва варіанти складніші в налаштуванні, ніж Ollama, і більше підходять для production/self-hosting сценаріїв.
- Порівняня цих методів з ollama: https://www.glukhov.org/llm-hosting/comparisons/docker-model-runner-vs-ollama-comparison/, https://www.glukhov.org/llm-hosting/comparisons/ollama-to-vllm-migration/  


## 1.4. Виклик API для отримання відповіді від моделі ollama 

In [ ]:
# Виконайте один раз, якщо бібліотеки ще не встановлені
%pip install -q datasets openai requests pandas python-dotenv

In [ ]:
import os
import re
import time
from pathlib import Path

import pandas as pd
import requests
from datasets import load_dataset
from dotenv import load_dotenv

In [ ]:
OLLAMA_MODEL = "qwen2.5:3b-instruct"  # <- за потреби змініть на вашу локальну модель
#OLLAMA_MODEL = "gemma3:12b" 

prompt = "Напиши вірш про кота, який грає на піаніно."
raw = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False},
        ).json()
print("======")
print("Повна JSON-відповідь від Ollama API:")
print(raw)
print("======")
print("фінальна відповідь від Ollama API:")
final_answer = raw.get("response", "").strip()
print(final_answer)
token_usage = {
            "input_tokens": raw.get("prompt_eval_count", 0),
            "output_tokens": raw.get("eval_count", 0),
        }
print("======")
print("Використання токенів від Ollama API:")
print(token_usage)

----
## 2 . Порівняння та вибір моделей


### Налаштування

Підключаємо ваш OpenAI API key. 
Свторимо файл .env і розмістимо там ключ. 
Приклад **.env.template**

Ціни OpenAI нижче задані як константи, бо тариф може змінюватися.
Поточні значення наведені для `gpt-5-mini` на момент підготовки ноутбука.


In [ ]:

from openai import OpenAI

# Ключ із файлу .env у корені проєкту (рядок OPENAI_API_KEY=...)
for _root in [Path.cwd(), *Path.cwd().parents]:
    _env = _root / ".env"
    if _env.is_file():
        load_dotenv(_env)
        break
else:
    load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "Немає OPENAI_API_KEY. Додайте .env у корінь проєкту або змінну середовища."
    )

OPENAI_MODEL = "gpt-5-mini"  # <- за потреби змініть модель OpenAI

# USD за 1M токенів (оновіть, якщо змінюєте модель OpenAI, ціни можна знайти тут: https://openai.com/pricing)
OPENAI_INPUT_PRICE = 0.25
OPENAI_OUTPUT_PRICE = 2.00

client = OpenAI()

-----
## 2.1. Бенчмарк exact-match: MMLU (`global_facts`)

Використовуємо **MMLU (`global_facts`)** з Hugging Face — тести з однозначною правильною відповіддю (A/B/C/D), тому accuracy рахується автоматично.

Датасет: https://huggingface.co/datasets/cais/mmlu

Інші популярні бенчмарки: MMLU-Pro, GSM8K, TruthfulQA, GPQA.

Порівнюємо якість, швидкість, надійність, пропускну здатність та вартість запитів. Це невелике технічне порівняння, а не повноцінна оцінка LLM.

![Compare exact match](Compare_exact_match.jpeg)

### Завантаження датасету та перегляд 3 випадкових питань 

In [ ]:

dataset = load_dataset("cais/mmlu", "global_facts", split="test")

print("Кількість питань:", len(dataset))
sample = dataset.to_pandas().sample(3, random_state=42).reset_index(drop=True)

for i, row in sample.iterrows():
    print(f"QUESTION {i + 1}")
    print(row["question"])
    for letter, choice in zip("ABCD", row["choices"]):
        print(f"{letter}. {choice}")
    print("Correct:", "ABCD"[row["answer"]])
    print("-" * 70)

### Формуємо prompt

Обидві моделі отримують однаковий prompt.

Просимо повернути **лише одну літеру** (A/B/C/D), бо тут оцінюємо правильність вибору, а не якість пояснення.
Так ми також зменшуємо затримку та вартість.


In [ ]:
def make_prompt(row):
    choices = "\n".join(
        f"{letter}. {choice}"
        for letter, choice in zip("ABCD", row["choices"])
    )

    return f"""Answer the multiple-choice question.
Return only one letter: A, B, C, or D.

Question:
{row["question"]}

{choices}
"""
# Як виглядає сформований промпт для одного з питань: 

prompt = make_prompt(sample.iloc[2])
print(prompt)


-----
## Learning: структуризація коду

На першій сесіії ми використовували деінде прямі виклики моделей. Але в реальному сценарії код швидко починає дублюватися, а підтримка ускладнюється: одна й та сама логіка з'являється в кількох місцях, її важче змінювати, тестувати та розширювати. Будь-яку правку доводиться вносити у всі дублікати вручну.

Якщо загорнути виклик у функцію, структура стає значно чистішою:
- основний сценарій читається легше;
- технічні деталі зосереджені в одному місці;
- зменшується дублювання коду;
- логіку простіше винести в окремий файл;
- моделі зручніше швидко перемикати для демо або порівняння з клієнтом.

Ключова ідея проста: зовні ми залишаємо один зрозумілий виклик, а відмінності між API різних провайдерів інкапсулюємо всередині функції.

----

### Виклики API моделей через єдину функцію
Нижче ми створимо універсальну функцію, яка виконує виклик потрібної моделі залежно від переданих параметрів.

----
#### Що робить функція `run_model`нижче

Ця функція приймає два параметри:
- `prompt` - текст, який ми надсилаємо моделі;
- `provider` - платформу моделі: `"ollama"` або `"openai"`.

Що вона робить всередині:
- відправляє запит у потрібний API;
- дістає фінальну текстову відповідь;
- дістає token usage;
- повертає все в одному однаковому форматі: `final_answer, token_usage`.

Завдяки цьому код нижче вже не залежить від деталей OpenAI або Ollama API і працює через один узгоджений інтерфейс.

Якщо згодом ви захочете додати, наприклад, Anthropic, достатньо буде додати ще одну гілку `if provider == "anthropic"`. При цьому решта ноутбука може залишитися без змін.

In [ ]:
def run_model(prompt, provider):
    if provider == "ollama":
        raw = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False},
        ).json()

        final_answer = raw.get("response", "").strip()
        token_usage = {
            "input_tokens": raw.get("prompt_eval_count", 0),
            "output_tokens": raw.get("eval_count", 0),
        }
        return final_answer, token_usage

    if provider == "openai":
        raw = client.responses.create(
            model=OPENAI_MODEL,
            input=prompt,
        )

        final_answer = raw.output_text.strip()
        token_usage = {
            "input_tokens": raw.usage.input_tokens,
            "output_tokens": raw.usage.output_tokens,
            "total_tokens": raw.usage.total_tokens,
        }
        return final_answer, token_usage

    raise ValueError("provider must be 'ollama' or 'openai'")

In [ ]:
# Один і той самий виклик для двох моделей
prompt = make_prompt(sample.iloc[0])

ollama_final_answer, ollama_token_usage = run_model(prompt, "ollama")
openai_final_answer, openai_token_usage = run_model(prompt, "openai")

print("Ollama final answer:", ollama_final_answer)
print("Ollama token_usage:", ollama_token_usage)
print("OpenAI final answer:", openai_final_answer)
print("OpenAI token_usage:", openai_token_usage)

----
### Довідка: що таке regexp (регулярний вираз)

**Regexp (regular expression)** — це короткий шаблон для пошуку/перевірки тексту за певним патерном, а не за точним збігом рядка. Такі патерни часто використовують для перевірки введених даних за завчасно відомим форматом: email, номер телефону, поштовий індекс тощо.

У Python для цього є вбудована бібліотека **`re`**.

У функції `parse_answer` нижче використовується патерн `r"\b([ABCD])\b"`:
- `\b` — межа слова (word boundary): гарантує, що літера йде окремо, а не як частина іншого слова (наприклад, не збіжиться з `A` у слові `Answer`).
- `[ABCD]` — клас символів: збігається з однією літерою A, B, C або D.
- `(...)` — група (capture group): дозволяє потім дістати саме цю знайдену літеру через `match.group(1)`.

`re.search(...)` шукає перший збіг патерну будь-де в тексті та повертає об'єкт `Match` (або `None`, якщо збігу немає) — саме тому в коді є перевірка `if match else None`.

Regex — доволі потужний інструмент перевірки й пошуку потрібних фрагментів у тексті, зі своєю власною мінімовою (спеціальні символи, класи, квантифікатори, групи тощо). Для тих, кому цікаво зануритись глибше, є зручні онлайн-симулятори, де можна писати патерн і одразу бачити збіги:
- https://regex101.com/
- https://regexr.com/


In [ ]:
def parse_answer(text):
    match = re.search(r"\b([ABCD])\b", str(text).upper()) #Regular expression to find a single letter A, B, C, or D in the text
    return match.group(1) if match else None

print("Ollama parsed:", parse_answer(ollama_final_answer))
print("OpenAI parsed:", parse_answer(openai_final_answer))

-----

## Повний exact-match бенчмарк

У цьому демо запускаємо "повний" бенчмарк: можна взяти всі 100 питань датасету або обрати меншу кількість. Зверніть увагу: для деяких локальних моделей повний прогін може зайняти значний час.

Нижче — один універсальний цикл для **однієї** обраної моделі (Ollama або OpenAI), що використовує функцію `run_model`. Запускати одразу кілька моделей і влаштовувати "змагання" між ними не потрібно: у нас вже є правильні (еталонні) відповіді з датасету, тому кожну модель порівнюємо саме з ними, а не одну з одною. Це і є суть **golden dataset** — заздалегідь відомого набору питань з правильними відповідями.

Щоб уникнути зайвих повторних запусків для тієї самої моделі, додано змінну **`run`**: якщо встановити її в `True`, бенчмарк виконається і результати збережуться на диск. Пізніше ці результати можна буде завантажити знову й оцінити точність моделі.


In [ ]:
# Повний прогін для однієї моделі + збереження в CSV


# Оберіть провайдера: "ollama" або "openai"
BENCH_PROVIDER = "ollama"
selected_model_name = OLLAMA_MODEL if BENCH_PROVIDER == "ollama" else OPENAI_MODEL

# Встановіть run=True для запуску
run = False

# Скільки питань взяти з датасету (максимум = len(dataset))
NUM_QUESTIONS = 100

if run:
    rows = []
    benchmark_started_at = time.perf_counter() # Початок вимірювання часу бенчмарку
    questions = dataset.select(range(min(NUM_QUESTIONS, len(dataset)))) #обираємо потрібну кількість питань з датасету

    for i, row in enumerate(questions):
        prompt = make_prompt(row) # Створюємо промпт для моделі на основі рядка з датасету
        correct = "ABCD"[row["answer"]]# Правильна відповідь з датасету

        t0 = time.perf_counter()
        final_answer, token_usage = run_model(prompt, BENCH_PROVIDER)
        parsed_answer = parse_answer(final_answer)

        rows.append({
            "model": selected_model_name,
            "question": row["question"],
            "correct_answer": correct,
            "model_answer": parsed_answer,
            "is_correct": parsed_answer == correct,
            "latency": time.perf_counter() - t0,
            "input_tokens": token_usage.get("input_tokens", 0),
            "output_tokens": token_usage.get("output_tokens", 0),
        })

        if (i + 1) % 10 == 0: #для відображення процессу бенчмарку виводимо кожні 10 питань
            print(BENCH_PROVIDER, i + 1, "/", len(questions))

    full_run_time_sec = time.perf_counter() - benchmark_started_at # Загальний час виконання бенчмарку

    #Зберігаємо результати в CSV
    results_dir = Path("results")
    results_dir.mkdir(exist_ok=True)
    benchmark_csv = results_dir / f"benchmark_{selected_model_name}_{NUM_QUESTIONS}.csv"

    all_results = pd.DataFrame(rows)
    all_results.to_csv(benchmark_csv, index=False)

    print(f"Saved: {benchmark_csv}")
    print(f"Rows: {len(all_results)}")
    print(f"Full run time (sec): {full_run_time_sec:.2f}")
else:
    print("run=False, бенчмарк не запускався.")

In [ ]:
# Якщо run=False, і дані вже існують, можна підвантажити останній CSV для обраного провайдера

results_dir = Path("results")
provider_files = sorted(results_dir.glob(f"benchmark_{selected_model_name}_*.csv"))

if not run:
    if not provider_files:
        raise FileNotFoundError(
            f"Немає CSV для model '{selected_model_name}' у '{results_dir}'."
        )

    latest_csv = provider_files[-1]
    all_results = pd.read_csv(latest_csv)
    print(f"Loaded: {latest_csv.name} ({len(all_results)} rows)")

all_results.tail()

## Підсумкові метрики

- **Accuracy:** Відсоток правильних відповідей на всі питання бенчмарку.

- **Median latency:** Типова тривалість запиту end-to-end. Медіана менш чутлива до поодиноких повільних запитів, ніж середнє.

- **Failure rate:** Відсоток запитів, де не вдалося отримати валідну відповідь A/B/C/D.

- **Output tokens/sec:** Проста метрика пропускної здатності: `output tokens / end-to-end latency`.

- **Median cost:** Типова вартість одного OpenAI-запиту. Для локальної Ollama показується `$0`, але це **не** означає нульову вартість заліза/електрики.

- **Full cost:** Сумарна вартість повного прогону через OpenAI API.


In [ ]:
# Порівняння кількох моделей: читаємо збережені результати з results/ за списком назв моделей
#MODELS_TO_COMPARE = [OLLAMA_MODEL, OPENAI_MODEL]
MODELS_TO_COMPARE = ["nemotron-3.5-lightning", "qwen2.5:3b-instruct","gpt-5-mini", "gpt-4o-mini", "gemma3:12b"]
results_dir = Path("results")
loaded_frames = []

for model_name in MODELS_TO_COMPARE:
    model_files = list(results_dir.glob(f"benchmark_{model_name}_*.csv"))
    if not model_files:
        print(f"Пропускаємо '{model_name}': немає збереженого CSV у '{results_dir}'.")
        continue

    csv_path = model_files[0] 
    loaded_frames.append(pd.read_csv(csv_path))
    print(f"Loaded: {csv_path.name}")

if not loaded_frames:
    raise FileNotFoundError(
        f"Немає жодного збереженого CSV для моделей {MODELS_TO_COMPARE} у '{results_dir}'."
    )

all_results = pd.concat(loaded_frames, ignore_index=True)

# Загальна кількість токенів (input + output) для кожного запиту
all_results["total_tokens"] = all_results["input_tokens"] + all_results["output_tokens"]

summary = (
    all_results
    .groupby("model") #Групуємо результати за моделлю
    .agg( #агрегуємо дані для отримання середніх значень
        accuracy=("is_correct", "mean"),
        mean_latency_sec=("latency", "mean"),
        mean_total_tokens=("total_tokens", "mean"),
    )
)

summary["accuracy"] = (summary["accuracy"] * 100).round(1)
summary["mean_latency_sec"] = summary["mean_latency_sec"].round(2)
summary["mean_total_tokens"] = summary["mean_total_tokens"].round(1)

summary


## Як інтерпретувати результат

> **Accuracy** показує, чи достатня якість локальної моделі для такого типу задач.

> **Latency** і **tokens** показують компроміс у швидкості та використанню токенів.


Наприклад, якщо локальна модель має майже таку саму точність, але трохи повільніша, вона все одно може бути вигідною, коли:

- дані мають залишатися локально;
- вартість API критична на масштабі;
- обмежений доступ до інтернету;
- важлива передбачувана інфраструктура.

Якщо OpenAI помітно точніша або швидша, хмарна модель може бути варта своєї ціни.

### Важливе обмеження

`MMLU/global_facts` — це лише **невеликий тест загальних знань**. Він не показує, яка модель буде кращою для RAG, витягання даних з документів, коду, сумаризації чи вашої бізнес-доменної задачі.

Для реального проєкту наступний крок — невеликий кастомний бенчмарк на реальних запитах користувачів.


----
----

## 2.2. LLM-as-a-Judge: міні-демо на Slim Orca Ukrainian

Ідея: порівняти дві відповіді (локальна модель vs OpenAI) не лише по exact-match, а й через механізм **LLM-as-a-Judge**.

Як це працює:
- беремо датасет `cidtd-mod-ua/slim-orca-ukrainian`;
- використовуємо лише **5 прикладів**;
- генеруємо відповіді обох моделей;
- окремий judge-модельний виклик порівнює відповіді за якістю.

Посилання на датасет:
https://huggingface.co/datasets/cidtd-mod-ua/slim-orca-ukrainian

![Compare with LLM as a Judge](Compare_LLM_as_a_judge.jpeg)

In [ ]:
# Міні-датасет для LLM-as-a-Judge: Slim Orca Ukrainian
judge_dataset = load_dataset("cidtd-mod-ua/slim-orca-ukrainian", split="train")
judge_sample = judge_dataset.shuffle(seed=250826).select(range(100))


# Беремо запит і референсну відповідь з базових полів
judge_items = [
    {
        "question": str(row.get("instruction") or row.get("prompt") or "").strip(),
        "context": str(row.get("input") or "").strip(),
        "reference_answer": str(row.get("output") or row.get("response") or "").strip(),
    }
    for row in judge_sample
]

# Друк 5 прикладів
for i, item in enumerate(judge_sample[:5], start=1):
    print(f"EXAMPLE {i}")
    item = judge_items[i - 1]
    print("Question:", item["question"][:1000])
    print("Context:", item["context"][:1000])
    print("Reference:", item["reference_answer"][:1000])
    print("-" * 80)

## Генеруємо відповіді моделі та одразу оцінюємо їх через Judge LLM

Аналогічно до розділу 2.1: обираємо одного провайдера (Ollama або OpenAI) і кількість питань.

Цього разу ми одразу (щоб зробити це за один прогон циклу) оцінюємо кожну відповідь через Judge LLM у момент її генерації, тому спочатку готуємо суддю (модель, метрики, функцію оцінки) — і лише потім запускаємо генерацію.

### Готуємо Judge LLM заздалегідь

Промпт і функція оцінки мають існувати до того, як почнеться генерація відповідей, бо кожну відповідь ми одразу оцінюємо після отримання.

In [ ]:
import json
import re

JUDGE_MODEL = "gpt-5"

# Метрики оцінки з описом критерію (легко розширити — додайте новий запис у словник).
# Кожна метрика — число від 0 до 1 (не лише 0/1): 1 = критерій виконано повністю,
# 0 = зовсім не виконано, проміжні значення — часткове виконання.
METRICS = {
    "all_facts_present": "яка частка ключових фактів з очікуваної відповіді присутня у відповіді моделі (1 = усі ключові факти присутні, 0 = жодного важливого факту немає; проміжне значення — якщо частина фактів відсутня, наприклад 0.5 якщо половина)",
    "similarity_to_reference": "наскільки відповідь моделі схожа за змістом на очікувану відповідь (1 = зміст повністю збігається, 0 = зміст повністю відрізняється; проміжне значення — якщо збігається лише частково)",
    "no_hallucination": "наскільки у відповіді відсутні вигадані або непідтверджені факти (1 = галюцинацій немає, 0 = відповідь майже повністю складається з вигаданих/непідтверджених фактів; проміжне значення — якщо є лише окремі непідтверджені деталі)",
    "human_style": "наскільки природно та зрозуміло звучить відповідь для людини (1 = природний, зрозумілий стиль, 0 = повністю незрозуміло або по-машинному; проміжне значення — якщо стиль частково незграбний)",
}

def score_answer(question, reference_answer, answer):
    criteria = "\n".join(f"- {metric}: {description}" for metric, description in METRICS.items())

    judge_prompt = f"""Ти неупереджений judge, який оцінює відповідь моделі відносно очікуваної відповіді.

Оціни відповідь за кожним критерієм нижче числом від 0 до 1 (не лише 0 або 1): 1 — критерій виконано повністю, 0 — зовсім не виконано, проміжні значення (наприклад 0.3, 0.5, 0.75) — критерій виконано частково.
{criteria}

Відповідь моделі може бути сформульована іншими словами, ніж очікувана відповідь, — це нормально. Оцінюй за суттю (зміст, факти), а не за дослівним збігом формулювання.

Поверни ТІЛЬКИ JSON з оцінками за всіма критеріями.

Запит:
{question}

Очікувана відповідь:
{reference_answer}

Відповідь моделі:
{answer}
"""

    judge_raw = client.responses.create(model=JUDGE_MODEL, input=judge_prompt)
    raw_text = judge_raw.output_text.strip()

    # Прагнемо парсити строго JSON; якщо модель додала текст навколо, дістаємо JSON-об'єкт regex'ом.
    # (Ці перевірки потрібні для стабільності: LLM іноді додає зайвий текст навколо JSON.)
    try:
        parsed = json.loads(raw_text)
    except Exception:
        match = re.search(r"\{[\s\S]*\}", raw_text)
        parsed = json.loads(match.group(0)) if match else {}

    # Значення приводимо до float і обмежуємо діапазоном [0, 1] на випадок, якщо модель вийде за межі.
    return {
        metric: round(min(1.0, max(0.0, float(parsed.get(metric, 0.0)))), 2)
        for metric in METRICS
    }


In [ ]:
# Генерація відповідей для однієї моделі + оцінка Judge LLM одразу + збереження в CSV

# Оберіть провайдера: "ollama" або "openai"
PROVIDER = "openai"
OPENAI_MODEL = "gpt-4o-mini"  # <- за потреби змініть модель OpenAI
model_name = OLLAMA_MODEL if PROVIDER == "ollama" else OPENAI_MODEL

# Встановіть run=True для запуску
run = False

# Скільки питань взяти з judge_items (максимум = len(judge_items))
NUM_QUESTIONS = 10

if run:
    judge_rows = []
    selected_items = judge_items[: min(NUM_QUESTIONS, len(judge_items))]

    for item in selected_items:
        question = item["question"]
        input_text = item["context"]
        reference = item["reference_answer"]

        qa_prompt = f"""{question}

    Context:
    {input_text}
    """

        t0 = time.perf_counter()
        final_answer, token_usage = run_model(qa_prompt, PROVIDER)
        latency = time.perf_counter() - t0

        scores = score_answer(question, reference, final_answer)

        judge_rows.append({
            "model": model_name,
            "question": question,
            "context": input_text,
            "reference_answer": reference,
            "answer": final_answer,
            "latency": round(latency, 2),
            "input_tokens": token_usage.get("input_tokens", 0),
            "output_tokens": token_usage.get("output_tokens", 0),
            **scores,
            "total_score": sum(scores.values()),
        })

    results_dir = Path("results")
    results_dir.mkdir(exist_ok=True)
    judge_csv = results_dir / f"judgebench_{model_name}_{NUM_QUESTIONS}.csv"

    judge_answers_df = pd.DataFrame(judge_rows)
    judge_answers_df.to_csv(judge_csv, index=False)

    print(f"Saved: {judge_csv}")
    print(f"Rows: {len(judge_answers_df)}")
else:
    print("run=False, генерація не запускалась.")

In [ ]:
# Якщо run=False, можна підвантажити останній CSV для обраної моделі
results_dir = Path("results")
judge_files = sorted(results_dir.glob(f"judgebench_{model_name}_*.csv"))

if not run:
    if not judge_files:
        raise FileNotFoundError(
            f"Немає CSV для моделі '{model_name}' у '{results_dir}'."
        )

    latest_judge_csv = judge_files[-1]
    judge_answers_df = pd.read_csv(latest_judge_csv)
    print(f"Loaded: {latest_judge_csv.name} ({len(judge_answers_df)} rows)")

judge_answers_df.tail()

## Порівнюємо моделі через Judge-based-benchmark

In [ ]:
# Порівняння моделей за оцінками Judge LLM (оцінки вже пораховані під час генерації, розділ 11)
#MODELS_FOR_JUDGE = [OLLAMA_MODEL, OPENAI_MODEL]
MODELS_FOR_JUDGE = ["nemotron-3.5-lightning", "qwen2.5:3b-instruct","gpt-5-mini","gpt-4o-mini", "gemma3:12b"]

results_dir = Path("results")
loaded_frames = []

for model_name in MODELS_FOR_JUDGE:
    model_files = list(results_dir.glob(f"judgebench_{model_name}_*.csv"))
    if not model_files:
        print(f"Пропускаємо '{model_name}': немає збереженого CSV у '{results_dir}'.")
        continue

    csv_path = model_files[0]  # для кожної моделі очікуємо рівно один збережений CSV
    loaded_frames.append(pd.read_csv(csv_path))
    print(f"Loaded: {csv_path.name}")

if not loaded_frames:
    raise FileNotFoundError(f"Немає жодного CSV для моделей {MODELS_FOR_JUDGE} у '{results_dir}'.")

all_judge_results = pd.concat(loaded_frames, ignore_index=True)

# Загальна кількість токенів (input + output) для кожного запиту
all_judge_results["total_tokens"] = all_judge_results["input_tokens"] + all_judge_results["output_tokens"]

judge_summary = (
    all_judge_results
    .groupby("model") #Групуємо результати за моделлю
    .agg( #агрегуємо дані для отримання середніх значень по кожній метриці Judge LLM + latency/tokens
        **{metric: (metric, "mean") for metric in METRICS},
        mean_latency_sec=("latency", "mean"),
        mean_total_tokens=("total_tokens", "mean"),
    )
)

for metric in METRICS:
    judge_summary[metric] = judge_summary[metric].round(2)
judge_summary["mean_latency_sec"] = judge_summary["mean_latency_sec"].round(2)
judge_summary["mean_total_tokens"] = judge_summary["mean_total_tokens"].round(1)

judge_summary


## Додаткові матеріали для самостійного вивчення (шукайте в додаткових варіантах): 
----
- Фреймворки для евалюації LLM/RAG
- Україномовні LLM: моделі та датасети для оцінки
- Leaderboards: актуальні порівняння 
---- 


## Чекліст вибору локальної LLM

- **Призначення:** чат, RAG-QA, кодинг, сумаризація, агентні задачі.
- **Мови (мультимовність):** якість саме для української + англійської.
- **Розмір моделі:** чи вистачає RAM/VRAM для вашого заліза.
- **Контекст:** чи вистачає вікна для system + chunks + питання.
- **Швидкість:** latency і токени/сек на вашій машині.
- **Стабільність формату:** чи стабільно повертає потрібний формат.
- **Безпека:** логи/кеш, зовнішні tool calls, потреба в VM/VPS.
- **Ліцензія:** перевірка умов комерційного використання.
- **TCO:** локальні витрати (залізо/електрика) vs API.

